# 01A: Exploring the Data Lake

Generic tooling to inspect the CICCADA data lake at three levels: **Storage** (S3 files), **Catalog** (Glue tables + schema), **Data** (actual rows). Reusable for any new data source.

**Three zoom levels:**

| Level | Question | Tool | Cost |
|---|---|---|---|
| **Storage** | What files physically exist? | `s3_ls()` (boto3) | free |
| **Catalog** | What's registered as a queryable table? | `databases()`, `tables()`, `describe()` (Glue) | free |
| **Data** | What do the values look like? | `aq()` (Athena) or `dread()` (DuckDB) | small / local |

Note: If anything says *"token expired"*, run `aws sso login --profile ciccada` in a terminal and re-run the cell.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))
sys.path.insert(0, str(pathlib.Path('lib').resolve()))

from shared.aws_config import *          # aq, dread, s3_ls, databases, tables
from shared.ciccada_config import SA, SAI, TABLES
import pandas as pd

## [Level 1] Storage: the physical files in S3

In [2]:
s3_ls()

,type,name,size_mb
0,folder,BOM_NCI/,None
1,folder,DeepArchive/,None
2,folder,Flask_App/,None
3,folder,PostgreJDBC_Driver/,None
4,folder,SAPN/,None
5,folder,SAPNTest/,None
6,folder,Trino-Warehouse/,None
7,folder,athena-results/,None
8,folder,spark-warehouse/,None
9,folder,temp_SolA/,None


In [3]:
s3_ls('Trino-Warehouse/solar_analytics/')

,type,name,size_mb
0,folder,Trino-Warehouse/solar_analytics/all_uncurtaile...,None
1,folder,Trino-Warehouse/solar_analytics/all_uncurtaile...,None
2,folder,Trino-Warehouse/solar_analytics/all_uncurtaile...,None
3,folder,Trino-Warehouse/solar_analytics/all_uncurtaile...,None
4,folder,Trino-Warehouse/solar_analytics/all_uncurtaile...,None
...,...,...,...
62,folder,Trino-Warehouse/solar_analytics/test_sola_2025...,None
63,folder,Trino-Warehouse/solar_analytics/ts-957d79213e8...,None
64,folder,Trino-Warehouse/solar_analytics/voltwatt_uncur...,None
65,folder,Trino-Warehouse/solar_analytics/voltwatt_uncur...,None


## [Level 2] Catalog: what's queryable, and its schema

Glue is the index that lets us write SQL. These calls are (metadata only).

In [4]:
databases()

,Database,Description
0,bom_nci,
1,default,Default Hive database
2,elb_logdb,
3,sapn2022,
4,solar_analytics,Migrated from Hive Metastore
5,solar_analytics_iceberg,
6,test_db,
7,type_probe,


In [5]:
# Every table in every database, in one view.
all_tabs = pd.concat([tables(d) for d in databases()['Database']], ignore_index=True)
all_tabs

,Database,Table,Description,TableType,Columns,Partitions
0,bom_nci,solar,,EXTERNAL_TABLE,"time, latitude, longitude, surface_global_irra...",
1,elb_logdb,elb_logs_tbl,ALB Access log table,EXTERNAL_TABLE,"type, time, elb, client_ip, client_port, targe...","year, month, day"
2,sapn2022,circuit_measurements,,EXTERNAL_TABLE,"c_id, utc_tstamp, energy, power, voltage, vmin...",
3,sapn2022,circuit_measurements_curtailment_train,,EXTERNAL_TABLE,"c_id, utc_tstamp, energy, power, voltage, vmin...",
4,solar_analytics,circuits,,EXTERNAL_TABLE,"site_id, device_id, circuit_id, device_type, c...",
5,solar_analytics,compliance_voltvar,,EXTERNAL_TABLE,"site_id, s_id, year, month, day, day_night, no...",
6,solar_analytics,compliance_voltwatt,,EXTERNAL_TABLE,"site_id, s_id, year, month, day, noncompliance...",
7,solar_analytics,meta_single_inverters,,EXTERNAL_TABLE,"circuit_id, site_id, device_id, device_type, c...",
8,solar_analytics,meta_single_inverters_wrong_capacity,,EXTERNAL_TABLE,"circuit_id, site_id, device_id, device_type, c...",
9,solar_analytics,meta_single_inverters_wrong_capacity_up2_3c,,EXTERNAL_TABLE,"circuit_id, site_id, device_id, device_type, c...",


In [6]:
# Peek at any table's schema (works on Iceberg via SELECT * LIMIT 1)
table = TABLES['conformance_voltvar']   # 'conformance_voltvar_v2'
# or directly
table = "meta_up23c"
aq(f'SELECT * FROM {table} LIMIT 5', database=SAI)

,site_id,state,postcode,longitude,latitude,dnsp_name,dc_capacity_kw,ac_capacity_kw,export_limit_kw,monitoring_start,...,m_id,avg_pf,std_pf,pf_99,pf_01,n_long,n_lat,distance_km,s_99,flex_export_detected
0,1233585204,VIC,3140,145.35,-37.755,Ausnet,5.10,5.0,NaN,2019-08-09,...,M17,0.998620,0.000225,0.999304,0.998327,145.34,-37.76,1.241018,4.614773,False
1,158521185,NSW,2488,153.55,-28.350,Essential,10.54,8.0,NaN,2019-10-31,...,M39,0.999972,0.000029,0.999994,0.999873,153.56,-28.36,1.569777,7.975964,False
2,2083958230,NSW,2763,150.90,-33.750,Endeavour,5.28,5.0,NaN,2020-01-07,...,M13,0.999996,0.000008,1.000000,0.999959,150.90,-33.74,1.110000,4.930158,True
3,248529252,VIC,3140,145.35,-37.755,Ausnet,5.06,5.0,NaN,2020-03-04,...,M13,0.991457,0.006766,0.997926,0.970164,145.34,-37.76,1.241018,3.980736,False
4,248529252,VIC,3140,145.35,-37.755,Ausnet,5.06,5.0,NaN,2020-03-04,...,M13,0.991457,0.006766,0.997926,0.970164,145.34,-37.76,1.241018,3.980736,False


## [Level 3] Data peek at actual rows

Two engines, same files. 

1. Use `aq()` (Athena) for normal SQL
2. use `dread()` (DuckDB, reads the Parquet file directly) for quick peeks or when Glue's description is wrong.

**Cost:** on the big `ts` table, always filter on `year`/`month`/`is_pv` and never sort on a fresh peek.

In [7]:
aq("SELECT * FROM ts WHERE is_pv = True AND year = 2024 AND month = 1 LIMIT 5", database=SAI)

,circuit_id,t_stamp,power,energy,energy_reactive,energy_import,energy_export,energy_reactive_import,energy_reactive_export,power_factor,voltage,current,year,month,is_pv,postcode
0,483249,2024-01-06 12:30:00,-1.2800,-0.1067,11.1700,0.0,0.1067,11.1700,0.0,0.000091,237.75,0.5745,2024,1,True,2570
1,483249,2024-01-06 12:45:00,-1.4400,-0.1200,11.2308,0.0,0.1200,11.2308,0.0,0.000114,238.55,0.5755,2024,1,True,2570
2,483249,2024-01-06 13:00:00,-1.5067,-0.1256,11.1953,0.0,0.1256,11.1953,0.0,0.000126,238.15,0.5745,2024,1,True,2570
3,483249,2024-01-06 13:05:00,-1.5300,-0.1275,11.1797,0.0,0.1275,11.1797,0.0,0.000130,238.10,0.5740,2024,1,True,2570
4,483249,2024-01-06 14:00:00,-1.4167,-0.1181,11.2528,0.0,0.1181,11.2528,0.0,0.000110,238.85,0.5760,2024,1,True,2570


## If Athena chokes: read the Parquet directly with DuckDB

Some results tables have *schema drift*. The Glue description disagrees with the file (e.g. a column the file stores as integer but Glue calls a double). This is common.

Athena won't be able to read it, so DuckDB reads the file's own schema and just works.

The S3 path comes from the error message, or from `s3_ls()`.

In [ ]:
# dread('s3://project-ciccada/Trino-Warehouse/solar_analytics/conformance_voltvar_v2/data/*.parquet')

## Recipe to explore any new data source in future

1. **`s3_ls('<prefix>/')`**: See what physically exists and how it's "foldered".
2. **`databases()` / `tables(db)`**: Check if it is registered in Glue. If yes, can use SQL.
3. **`describe('<table>', db)`**: Learn its columns.
4. **`aq('SELECT ... LIMIT 5')`**: Peek at values (filter on partitions if it's big).
5. Note: Not in Glue, or Glue is wrong?: Point **`dread('s3://.../*.parquet')`** straight at the files.